# Stage B.1 — Fit listener models to effectiveness ratings

Reproduces the **9-cell effectiveness-fit analysis** from the paper. For each
combination of `speaker_condition × listener_belief_condition` (3 × 3 = 9
cells), fits five listener models — `literal`, `credulous_T`, `credulous_F`,
`vigilant_T`, `vigilant_F` (where `_T`/`_F` is `update_internal` True/False) —
to participants' effectiveness ratings, optimizing the rationality
parameter α independently for each pragmatic model.

Expected ranges (from paper):
- Informative-speaker × literal listener:   mean LL ∈ [-10.10, -8.11]
- Persuasive-speaker × credulous listener:  mean LL ∈ [-14.33, -13.70]

The headline finding: literal best for informative; credulous best for
persuasive; differences typically < 1 LL — effectiveness ratings alone don't
discriminate listener models well, and explicit warning about biased speakers
(vigilant condition) doesn't substantially change estimates.

## Setup — imports, configuration, data load

In [1]:
import numpy as np
import pandas as pd
import copy
from time import time
from rsa_optimal_exp_core import World, LiteralListener, PragmaticListener_obs_n


UTTERANCE_SEQUENCES = {
    "informative": [
        ["most,successful", "no,unsuccessful", "all,successful", "no,unsuccessful", "all,successful"],
        ["most,unsuccessful", "all,unsuccessful", "no,successful", "all,unsuccessful", "no,successful"],
    ],
    "pers_plus": [
        ["some,successful", "some,successful", "some,unsuccessful", "some,successful", "some,successful"],
        ["most,successful", "some,successful", "some,unsuccessful", "some,successful", "some,unsuccessful"],
        ["most,successful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,successful"],
    ],
    "pers_minus": [
        ["some,unsuccessful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,unsuccessful"],
        ["most,unsuccessful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,successful"],
        ["most,unsuccessful", "some,successful", "some,unsuccessful", "some,successful", "some,unsuccessful"],
    ],
}

THETA_VALUES = np.linspace(0, 1, 21)             # 0%, 5%, ..., 100%
ALPHA_GRID = np.logspace(np.log10(1), np.log10(50), 100)   # 100 log-spaced α values
EPSILON = 0.01                                   # smoothing floor for log(0) avoidance
N_THETA = len(THETA_VALUES)

LISTENER_CONDITIONS = ["credulous", "vigilant", "naturalistic"]
SPEAKER_CONDITIONS = ["informative", "pers_plus", "pers_minus"]
PRAGMATIC_MODELS = ["credulous_T", "credulous_F", "vigilant_T", "vigilant_F"]
ALL_MODELS = ["literal"] + PRAGMATIC_MODELS

INPUT_PATH = "./processed_listener_n1_anonymized.csv"


/Users/kangke/projects/prag_net/data/cogsci_rsa_listener_experiment_n5_o1/rsa_optimal_exp_core.py:2049: SyntaxWarning: invalid escape sequence '\p'
  Returns P(theta) \propto exp( sum_{psi, alpha} log_joint(theta,psi,alpha) ).
/Users/kangke/projects/prag_net/data/cogsci_rsa_listener_experiment_n5_o1/rsa_optimal_exp_core.py:2062: SyntaxWarning: invalid escape sequence '\p'
  Returns P(psi) \propto exp( sum_{theta, alpha} log_joint(theta,psi,alpha) ).
/Users/kangke/projects/prag_net/data/cogsci_rsa_listener_experiment_n5_o1/rsa_optimal_exp_core.py:2072: SyntaxWarning: invalid escape sequence '\p'
  Returns P(alpha) \propto exp( sum_{theta, psi} log_joint(theta,psi,alpha)).


## Build the model-prediction lookup table

For each `(speaker_cond, sequence_idx)` × model × α, simulate the listener
through the 5-utterance sequence and record the posterior over θ at each
round. Heavy compute — ~1 minute, runs once.

In [2]:
world = World(n=1, m=5, theta_values=THETA_VALUES)

lookup = {}
t0 = time()
for speaker_cond, sequences in UTTERANCE_SEQUENCES.items():
    for seq_idx, utterances in enumerate(sequences):
        key = (speaker_cond, seq_idx)
        lookup[key] = {"utterances": utterances, "literal": None}
        for m in PRAGMATIC_MODELS:
            lookup[key][m] = {}

        # Literal listener — no α to fit.
        lit = LiteralListener(world)
        post = []
        for u in utterances:
            lit.listen_and_update(u)
            post.append(lit.current_belief_theta.copy())
        lookup[key]["literal"] = np.asarray(post)

        # Pragmatic models × α grid.
        for model_name, omega, update_internal in [
            ("credulous_T", "coop", True),  ("credulous_F", "coop", False),
            ("vigilant_T", "strat", True),  ("vigilant_F", "strat", False),
        ]:
            for alpha in ALPHA_GRID:
                pl = PragmaticListener_obs_n(
                    world=world, level=1, omega=omega,
                    update_internal=update_internal, alpha=alpha, beta=0.0,
                )
                post = []
                for u in utterances:
                    pl.listen_and_update(u)
                    post.append(pl.current_belief_theta.copy())
                lookup[key][model_name][alpha] = np.asarray(post)
        print(f"  {key}: lookup built ({time()-t0:.1f}s elapsed)")

# Smooth predictions to avoid log(0) when an empirical response sits at
# a θ-bin where the model assigns ~zero probability.
smoothed_lookup = copy.deepcopy(lookup)
for key in smoothed_lookup:
    smoothed_lookup[key]["literal"] = (1 - EPSILON) * smoothed_lookup[key]["literal"] + EPSILON / N_THETA
    for m in PRAGMATIC_MODELS:
        for alpha in ALPHA_GRID:
            smoothed_lookup[key][m][alpha] = (1 - EPSILON) * smoothed_lookup[key][m][alpha] + EPSILON / N_THETA
print(f"\nLookup tables ready in {time()-t0:.1f}s")


/Users/kangke/projects/prag_net/data/cogsci_rsa_listener_experiment_n5_o1/rsa_optimal_exp_core.py:382: UserWarning: theta values above precision is rounded to 10 decimals
  warnings.warn("theta values above precision is rounded to 10 decimals",


  ('informative', 0): lookup built (9.9s elapsed)
  ('informative', 1): lookup built (17.9s elapsed)
  ('pers_plus', 0): lookup built (26.3s elapsed)
  ('pers_plus', 1): lookup built (33.4s elapsed)
  ('pers_plus', 2): lookup built (41.9s elapsed)
  ('pers_minus', 0): lookup built (49.6s elapsed)
  ('pers_minus', 1): lookup built (57.3s elapsed)
  ('pers_minus', 2): lookup built (64.9s elapsed)

Lookup tables ready in 65.0s


## Load and filter participant data

Per the paper, participants are filtered by completion + attention check
only (the comprehension-pass filters are not applied — they shift LLs by
~0.5 but don't change the qualitative pattern).

In [3]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} participants")

df = df[(df["completed_all_rounds"] == True) & (df["attention_check_passed"] == True)].copy()
print(f"After completed + attention-passed: {len(df)} participants")

response_cols = [f"r{i}_effectiveness" for i in range(1, 6)]
df = df.dropna(subset=["speaker_condition", "listener_belief_condition", "sequence_idx"] + response_cols)
df["sequence_idx"] = df["sequence_idx"].astype(int)
for c in response_cols:
    df[c] = df[c].astype(int)
print(f"After NaN drop: {len(df)} participants")

# Convert 0-100 effectiveness ratings to 0-20 indices on the θ grid (step=5%).
responses = df[response_cols].values
response_indices = np.clip(responses // 5, 0, N_THETA - 1).astype(int)
print(f"Response matrix shape: {response_indices.shape}")

print("\nN per (speaker × listener) cell:")
print(df.groupby(["speaker_condition", "listener_belief_condition"]).size())


Loaded 634 participants
After completed + attention-passed: 609 participants
After NaN drop: 609 participants
Response matrix shape: (609, 5)

N per (speaker × listener) cell:
speaker_condition  listener_belief_condition
informative        credulous                    68
                   naturalistic                 72
                   vigilant                     59
pers_minus         credulous                    52
                   naturalistic                 64
                   vigilant                     67
pers_plus          credulous                    76
                   naturalistic                 75
                   vigilant                     76
dtype: int64


## Fit α per (speaker_cond × listener_belief_cond) cell

For each of 9 cells: literal LL needs no α; for each pragmatic model,
sweep `ALPHA_GRID` and pick α maximizing the *mean* LL across participants
in that cell.

In [4]:
ll_by_cell = {}
best_alphas = {}
mean_lls = {}

for sc in SPEAKER_CONDITIONS:
    for lc in LISTENER_CONDITIONS:
        cell = (sc, lc)
        sub = df[(df["speaker_condition"] == sc) & (df["listener_belief_condition"] == lc)]
        if len(sub) == 0:
            continue
        sub_pos = [df.index.get_loc(i) for i in sub.index]
        ll_by_cell[cell] = {}
        mean_lls[cell] = {}

        # Literal — no α.
        lits = []
        for ii in sub_pos:
            row = df.iloc[ii]
            key = (row["speaker_condition"], int(row["sequence_idx"]))
            resp_idx = response_indices[ii]
            probs = smoothed_lookup[key]["literal"][np.arange(5), resp_idx]
            lits.append(np.sum(np.log(probs)))
        ll_by_cell[cell]["literal"] = lits
        mean_lls[cell]["literal"] = float(np.mean(lits))

        # Pragmatic models — fit α per cell.
        for m in PRAGMATIC_MODELS:
            best_mean = -np.inf
            best_alpha = None
            best_lls = None
            for alpha in ALPHA_GRID:
                lls = []
                for ii in sub_pos:
                    row = df.iloc[ii]
                    key = (row["speaker_condition"], int(row["sequence_idx"]))
                    resp_idx = response_indices[ii]
                    probs = smoothed_lookup[key][m][alpha][np.arange(5), resp_idx]
                    lls.append(np.sum(np.log(probs)))
                mean_ll = float(np.mean(lls))
                if mean_ll > best_mean:
                    best_mean = mean_ll
                    best_alpha = alpha
                    best_lls = lls
            ll_by_cell[cell][m] = best_lls
            mean_lls[cell][m] = best_mean
            best_alphas[(cell, m)] = best_alpha

# Display results table
print("Mean LL per (speaker × listener) per model")
print("=" * 90)
print(f"{'cell':<35} {'literal':>9} {'cred_T':>9} {'cred_F':>9} {'vig_T':>9} {'vig_F':>9}  best")
for sc in SPEAKER_CONDITIONS:
    for lc in LISTENER_CONDITIONS:
        cell = (sc, lc)
        if cell not in mean_lls:
            continue
        ml = mean_lls[cell]
        winner = max(ml, key=ml.get)
        n = len(ll_by_cell[cell]["literal"])
        print(f"({sc:13s}, {lc:13s}, n={n:3d})  "
              + "  ".join(f"{ml[m]:>7.2f}" for m in ALL_MODELS) + f"  {winner}")


Mean LL per (speaker × listener) per model
cell                                  literal    cred_T    cred_F     vig_T     vig_F  best
(informative  , credulous    , n= 68)    -7.91    -9.03    -9.03    -8.41    -8.52  literal
(informative  , vigilant     , n= 59)   -10.31   -11.26   -11.26   -10.73   -10.83  literal
(informative  , naturalistic , n= 72)    -8.38    -9.43    -9.43    -8.85    -8.96  literal
(pers_plus    , credulous    , n= 76)   -14.74   -15.06   -14.12   -14.46   -14.50  credulous_F
(pers_plus    , vigilant     , n= 76)   -14.57   -14.72   -13.99   -14.66   -14.55  credulous_F
(pers_plus    , naturalistic , n= 75)   -14.82   -15.22   -14.40   -14.68   -14.69  credulous_F
(pers_minus   , credulous    , n= 52)   -14.19   -14.42   -13.79   -14.23   -14.16  credulous_F
(pers_minus   , vigilant     , n= 67)   -14.67   -14.90   -14.18   -14.65   -14.62  credulous_F
(pers_minus   , naturalistic , n= 64)   -14.08   -14.24   -13.57   -14.08   -14.06  credulous_F
